# explore and preprocess ybt for testing and comparison w/ c4

# import and load data
- still broken and doesnt work needs debugging

In [45]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/raw/YBT.csv', header=1)
print("Shape after loading:", df.shape)
print(df.head(10))
print(df.columns.tolist())

df.columns = df.columns.str.strip()
print(df.columns.tolist())


Shape after loading: (24204, 83)
                  Progress    Duration (in seconds)                 Finished  \
0  {"ImportId":"progress"}  {"ImportId":"duration"}  {"ImportId":"finished"}   
1                        1                       88                    FALSE   
2                      100                     4467                     TRUE   
3                      100                     2013                     TRUE   
4                      100                      902                     TRUE   
5                      100                     7571                     TRUE   
6                        1                        5                    FALSE   
7                        1                       38                    FALSE   
8                       30                      580                    FALSE   
9                       30                     1322                    FALSE   

                                       Recorded Date  \
0  {"ImportId":"recordedDate",

/var/folders/1b/r6y9_1zx175chs6rm5vj_xfc0000gp/T/ipykernel_76210/1360455978.py:5: DtypeWarning: Columns (0,1,2,31,32,33,34,35,36,37,38,39,48,49,80,81,82) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/raw/YBT.csv', header=1)


# impute demographics and questionnaire scores

In [34]:
demographic_cols = ['sex', 'handedness', 'education', 'occupation', 'country']
for col in demographic_cols:
    if col in df.columns:
        df[col] = df[col].fillna('unknown')

# Convert questionnaire columns to numeric and impute with median
questionnaire_cols = [col for col in df.columns if any(q in col for q in ['spq_', 'eq10_', 'sq10_', 'aq_'])]
df[questionnaire_cols] = df[questionnaire_cols].apply(pd.to_numeric, errors='coerce')
df[questionnaire_cols] = df[questionnaire_cols].fillna(df[questionnaire_cols].median())
print("Shape after loading:", df.shape)


Shape after loading: (24205, 83)


# drop rows with missing q data 

In [35]:
# print actual q columns and their data
print("Questionnaire columns:", questionnaire_cols)
print(df[questionnaire_cols].head(10))
print(df[questionnaire_cols].isnull().sum())
print("Shape after loading:", df.shape)


Questionnaire columns: ['eq10_1', 'eq10_2', 'eq10_3', 'eq10_4', 'eq10_5', 'eq10_6', 'eq10_7', 'eq10_8', 'eq10_9', 'eq10_10', 'sq10_1', 'sq10_2', 'sq10_3', 'sq10_4', 'sq10_5', 'sq10_6', 'sq10_7', 'sq10_8', 'sq10_9', 'sq10_10', 'aq_1', 'aq_2', 'aq_3', 'aq_4', 'aq_5', 'aq_6', 'aq_7', 'aq_8', 'aq_9', 'aq_10']
   eq10_1  eq10_2  eq10_3  eq10_4  eq10_5  eq10_6  eq10_7  eq10_8  eq10_9  \
0     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
1     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
2     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
3     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
4     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
5     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
6     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   
7     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN     NaN   


# map likert responses to numeric values

In [40]:
print(df[questionnaire_cols].head(10))
print(df[questionnaire_cols].dtypes)
print(df['eq10_1'].unique())

Empty DataFrame
Columns: [eq10_1, eq10_2, eq10_3, eq10_4, eq10_5, eq10_6, eq10_7, eq10_8, eq10_9, eq10_10, sq10_1, sq10_2, sq10_3, sq10_4, sq10_5, sq10_6, sq10_7, sq10_8, sq10_9, sq10_10, aq_1, aq_2, aq_3, aq_4, aq_5, aq_6, aq_7, aq_8, aq_9, aq_10]
Index: []

[0 rows x 30 columns]
eq10_1     int64
eq10_2     int64
eq10_3     int64
eq10_4     int64
eq10_5     int64
eq10_6     int64
eq10_7     int64
eq10_8     int64
eq10_9     int64
eq10_10    int64
sq10_1     int64
sq10_2     int64
sq10_3     int64
sq10_4     int64
sq10_5     int64
sq10_6     int64
sq10_7     int64
sq10_8     int64
sq10_9     int64
sq10_10    int64
aq_1       int64
aq_2       int64
aq_3       int64
aq_4       int64
aq_5       int64
aq_6       int64
aq_7       int64
aq_8       int64
aq_9       int64
aq_10      int64
dtype: object
[]


In [39]:
response_mapping = {
    'strongly disagree': 1,
    'slightly disagree': 2,
    'slightly agree': 3,
    'strongly agree': 4,
}
for col in questionnaire_cols:
    df[col] = df[col].map(response_mapping)
print("Shape after loading:", df.shape)


Shape after loading: (0, 83)


In [37]:
# Optionally, drop rows with too many missing questionnaire items (e.g., more than 2 missing)
df = df.dropna(subset=questionnaire_cols, how='all')
print("After dropping rows with all missing questionnaire data:", df.shape)
print("After dropping rows with too many missing questionnaire items:", df.shape)
print("Shape after loading:", df.shape)


After dropping rows with all missing questionnaire data: (0, 83)
After dropping rows with too many missing questionnaire items: (0, 83)
Shape after loading: (0, 83)


# standardize all questionnaire features 

In [38]:
scaler = StandardScaler()
if questionnaire_cols:
    df[questionnaire_cols] = scaler.fit_transform(df[questionnaire_cols])
print("Standardized questionnaire features.")

ValueError: Found array with 0 sample(s) (shape=(0, 30)) while a minimum of 1 is required by StandardScaler.

# aggregate and feature engineering

In [ ]:
print(df.columns.tolist())

['{"ImportId":"progress"}', '{"ImportId":"duration"}', '{"ImportId":"finished"}', '{"ImportId":"recordedDate","timeZone":"America/Denver"}', '{"ImportId":"_recordId"}', '{"ImportId":"QID271"}', '{"ImportId":"QID272"}', '{"ImportId":"QID273"}', '{"ImportId":"QID274"}', '{"ImportId":"QID275"}', '{"ImportId":"QID278"}', '{"ImportId":"QID279"}', '{"ImportId":"QID281"}', '{"ImportId":"QID282"}', '{"ImportId":"QID282_69_TEXT"}', '{"ImportId":"QID864"}', '{"ImportId":"QID314"}', '{"ImportId":"QID315"}', '{"ImportId":"QID316"}', '{"ImportId":"QID317"}', '{"ImportId":"QID318"}', '{"ImportId":"QID319"}', '{"ImportId":"QID320"}', '{"ImportId":"QID322"}', '{"ImportId":"QID323"}', '{"ImportId":"QID276"}', '{"ImportId":"QID808"}', '{"ImportId":"QID832"}', '{"ImportId":"QID832_30_TEXT"}', '{"ImportId":"QID868"}', '{"ImportId":"QID868_30_TEXT"}', '{"ImportId":"QID870"}', '{"ImportId":"QID870_1_TEXT"}', '{"ImportId":"QID871"}', '{"ImportId":"QID871_1_TEXT"}', '{"ImportId":"QID872"}', '{"ImportId":"QID8

In [ ]:
# Aggregate scores
df['eq_total'] = df[[f'eq10_{i}' for i in range(1, 11)]].sum(axis=1)
df['sqr_total'] = df[[f'sq10_{i}' for i in range(1, 11)]].sum(axis=1)
df['aq_total'] = df[[f'aq_{i}' for i in range(1, 11)]].sum(axis=1)
df['d_score'] = df['eq_total'] - df['sqr_total']

# Individual EQ items (for model compatibility)
for i in [1, 3, 6, 7, 10]:
    df[f'eq_{i}'] = df[f'eq10_{i}']

# Sex to numeric for interaction
sex_map = {'male': 0, 'female': 1, 'other': 2, 'prefer_not_to_say': 3, 'unknown': 4}
df['sex_num'] = df['sex'].map(sex_map).fillna(4)
df['sex_x_eq'] = df['sex_num'] * df['eq_total']

# Age to numeric and interaction
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df['age_x_aq'] = df['age'] * df['aq_total']

# SPQ features (not in YBT)
df['spq_total'] = 0
df['spq_5'] = 0

# Ensure all required AQ items exist
for i in [4, 5, 6, 8, 9, 10]:
    if f'aq_{i}' not in df.columns:
        df[f'aq_{i}'] = 0

KeyError: "None of [Index(['eq10_1', 'eq10_2', 'eq10_3', 'eq10_4', 'eq10_5', 'eq10_6', 'eq10_7',\n       'eq10_8', 'eq10_9', 'eq10_10'],\n      dtype='object')] are in the [columns]"